# Exploration 9 — Preprocess Stage 2
GPU-accelerated, resumable. Inputs are staged to local Colab SSD; durable caches stay on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os

PUMA_ROOT = Path('/content/drive/MyDrive/Research/PUMA')

PROJECT_ROOT = PUMA_ROOT / 'Exploration_9_Natural_Transfer_Validation'
DATASET_ROOT = PUMA_ROOT / 'Dataset'
UNI2_CHECKPOINT = PUMA_ROOT / 'PUMA_pretrained_checkpoints' / 'uni2_h_model.bin'

PREPROCESSED_ROOT = PROJECT_ROOT / 'PREPROCESSED'
PREP_DIR = PREPROCESSED_ROOT / 'prepared'
FEATURE_DIR = PREPROCESSED_ROOT / 'features'
LOCAL_STAGE = Path('/content/e9_preprocess_stage')
CPU_CORES = os.cpu_count() or 1

for path in (PROJECT_ROOT, DATASET_ROOT, UNI2_CHECKPOINT):
    if not path.exists():
        raise FileNotFoundError(path)

PREP_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
print('CPU cores:', CPU_CORES)


In [ ]:
import os, shutil

# Stage only the files used repeatedly by Tier-A / UNI2 to local SSD.
# Context/tissue folders stay on Drive via symlink; they are only provenance/audit inputs.
LOCAL_DATASET = LOCAL_STAGE / 'PUMA_DATASET'
LOCAL_CHECKPOINT = LOCAL_STAGE / 'uni2_h_model.bin'
LOCAL_STAGE.mkdir(parents=True, exist_ok=True)
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

for name in ('01_training_dataset_tif_ROIs', '01_training_dataset_geojson_nuclei'):
    src = DATASET_ROOT / name
    dst = LOCAL_DATASET / name
    marker = dst / '.STAGED_COMPLETE'
    if not marker.is_file():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        marker.write_text('complete')

for name in ('01_training_dataset_tif_context_ROIs', '01_training_dataset_geojson_tissue'):
    src = DATASET_ROOT / name
    dst = LOCAL_DATASET / name
    if not dst.exists():
        dst.symlink_to(src, target_is_directory=True)

if (not LOCAL_CHECKPOINT.is_file()) or LOCAL_CHECKPOINT.stat().st_size != UNI2_CHECKPOINT.stat().st_size:
    shutil.copy2(UNI2_CHECKPOINT, LOCAL_CHECKPOINT)

print('Local dataset:', LOCAL_DATASET)
print('Local UNI2:', LOCAL_CHECKPOINT)


In [ ]:
import os, sys, subprocess
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Crop/Tier-A parallelism uses all logical cores. Keep native kernels single-threaded
# inside each worker so CPU cores are not oversubscribed.
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
torch.set_float32_matmul_precision('highest')
torch.backends.cuda.matmul.allow_tf32 = False
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
import subprocess

subprocess.run([
    sys.executable, '-m', 'CODE.CLI.prepare_dataset',
    '--dataset-root', str(LOCAL_DATASET),
    '--output-dir', str(PREP_DIR),
    '--exploratory-all-official-train',
    '--workers', '0',
], check=True)

subprocess.run([
    sys.executable, '-m', 'CODE.CLI.extract',
    '--manifest', str(PREP_DIR / 'e9_natural_train_roster.csv'),
    '--checkpoint', str(LOCAL_CHECKPOINT),
    '--output-dir', str(FEATURE_DIR),
    '--device', 'cuda',
    '--microbatch', 'auto',
    '--max-microbatch', '0',
    '--cpu-workers', '0',
    '--features', 'local', 'wide',
], check=True)


In [ ]:
import json
status = {
    'status': 'PREPROCESS_COMPLETE',
    'prepared_dir': str(PREP_DIR),
    'feature_dir': str(FEATURE_DIR),
}
(PREPROCESSED_ROOT / 'PREPROCESS_STATUS.json').write_text(json.dumps(status, indent=2))
print(status)


In [ ]:
from google.colab import runtime
runtime.unassign()